In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

T_trans = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
T_locations = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")
T_date = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_date")
T_film = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_film")

In [0]:
# ── Cinema transactions (filtered to your specific titles) ────────────────────
df_cinema = (
    T_trans
    .join(T_locations.select("LocationID", "LC_Location_Type_Description", "LC_Trade_Brand_Description"), T_trans.LocationId == T_locations.LocationID, how='left')
    .join(T_date.select("dateid", "datefull"), T_trans.VisitDateId == T_date.dateid, how='left')
    .join(T_film.select("film_id", "title_id", "title_name"), T_film.film_id == T_trans.film_id, how='left')
    .filter(F.col("title_id").isin('38504', '38783', '35589', '37354', '36278'))
    .filter(F.col("LC_Location_Type_Description") == "Cineplex Theatre")
    .filter(F.col("SCENEFLAG") == "Yes")
    .filter(F.col("CDE_ID").isNotNull())
    .filter(F.col("datefull").between(F.add_months(F.current_date(), -24), F.current_date()))
    .withColumn("visit_key", F.concat_ws("|", "visitdateid", T_trans.LocationId, "CDE_ID"))
    .withColumn("Month", F.month("datefull"))
    .withColumn("Year", F.year("datefull"))
    .select("CDE_ID", "datefull", "Month", "Year", "visit_key",
            "LC_Trade_Brand_Description", "TicketFLAG", "ConcessionFLAG",
            "Quantity", "Net_Revenue", "Net_Spend", "title_name",
            T_trans.LocationId.alias("LocationId"))
)

# ── LBE transactions for the same members only ────────────────────────────────
cinema_members = df_cinema.select("CDE_ID").distinct()

df_lbe = (
    T_trans
    .join(T_locations.select("LocationID", "LC_Location_Type_Description", "LC_Trade_Brand_Description"), T_trans.LocationId == T_locations.LocationID, how='left')
    .join(T_date.select("dateid", "datefull"), T_trans.VisitDateId == T_date.dateid, how='left')
    .filter(F.col("LC_Location_Type_Description") == "Restaurant/Store/Games")
    .filter(F.col("SCENEFLAG") == "Yes")
    .filter(F.col("CDE_ID").isNotNull())
    .filter(F.col("datefull").between(F.add_months(F.current_date(), -24), F.current_date()))
    .join(cinema_members, on="CDE_ID", how="inner")
    .withColumn("visit_key", F.concat_ws("|", "visitdateid", T_trans.LocationId, "CDE_ID"))
    .withColumn("Month", F.month("datefull"))
    .withColumn("Year", F.year("datefull"))
    .withColumn("title_name", F.lit(None).cast("string"))
    .select("CDE_ID", "datefull", "Month", "Year", "visit_key",
            "LC_Trade_Brand_Description", "TicketFLAG", "ConcessionFLAG",
            "Quantity", "Net_Revenue", "Net_Spend", "title_name",
            T_trans.LocationId.alias("LocationId"))
)

df_all_trans = df_cinema.union(df_lbe)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

# ── Period labels ─────────────────────────────────────────────────────────────
def add_period(df):
    return df.withColumn("period",
        F.when(F.col("datefull").between("2024-07-01", "2024-09-30"), F.lit("July 2024 – Sep 2024"))
         .when(F.col("datefull").between("2025-07-01", "2025-09-30"), F.lit("July 2025 – Sep 2025"))
    ).filter(F.col("period").isNotNull())

period_start = F.when(F.col("period") == "July 2024 – Sep 2024", F.lit("2024-07-01").cast("date")) \
                .when(F.col("period") == "July 2025 – Sep 2025", F.lit("2025-07-01").cast("date"))

# ── Split into cinema and LBE pools (keep LocationId on both) ─────────────────
df_cinema_anchor = add_period(
    df_all_trans.filter(F.col("title_name").isNotNull())
).select("CDE_ID", "datefull", "title_name", "period",
         F.col("LocationId").alias("cinema_location_id"))

df_lbe_pool = add_period(
    df_all_trans.filter(F.col("title_name").isNull())
).withColumn("p_start", period_start)

# ── Join: cinema first, then LBE in same period ───────────────────────────────
df_cross = (
    df_lbe_pool
    .join(
        df_cinema_anchor.select(
            F.col("CDE_ID").alias("cin_CDE_ID"),
            F.col("datefull").alias("cinema_date"),
            F.col("title_name").alias("cin_title_name"),
            F.col("period").alias("cin_period"),
            F.col("cinema_location_id")
        ),
        (df_lbe_pool.CDE_ID   == F.col("cin_CDE_ID")) &
        (df_lbe_pool.period   == F.col("cin_period")) &
        (F.col("cinema_date") >= df_lbe_pool.p_start) &
        (F.col("cinema_date") <  df_lbe_pool.datefull),
        how="inner"
    )
    .withColumn("days_gap", F.datediff("datefull", "cinema_date"))
)

# ── Keep nearest prior cinema visit per LBE visit ────────────────────────────
w = Window.partitionBy("CDE_ID", "datefull", "LC_Trade_Brand_Description", "cin_title_name", "period")

df_cross_nearest = (
    df_cross
    .withColumn("min_days", F.min("days_gap").over(w))
    .filter(F.col("days_gap") == F.col("min_days"))
    .dropDuplicates(["CDE_ID", "datefull", "LC_Trade_Brand_Description", "cin_title_name", "period"])
    .withColumn("days_bucket",
        F.when(F.col("days_gap") <= 10, F.lit("01 – 10 days"))
         .when(F.col("days_gap") <= 15, F.lit("11 – 15 days"))
         .when(F.col("days_gap") <= 20, F.lit("16 – 20 days"))
         .when(F.col("days_gap") <= 25, F.lit("21 – 25 days"))
         .when(F.col("days_gap") <= 30, F.lit("26 – 30 days"))
         .otherwise(F.lit("> 30 days"))
    )
)

# ── Cross-visit metrics with day buckets ──────────────────────────────────────
df_cross_agg = (
    df_cross_nearest
    .groupBy("period", "LC_Trade_Brand_Description", "cin_title_name", "days_bucket")
    .agg(
        F.countDistinct("CDE_ID").alias("Cross_Visitors"),
        F.round(F.avg("days_gap"), 1).alias("Avg_Days_Cinema_to_LBE")
    )
)

# ── Cinema title metrics (total viewers per period + title) ───────────────────
df_cinema_agg = (
    df_cinema_anchor
    .groupBy("period", "title_name")
    .agg(F.countDistinct("CDE_ID").alias("Title_Viewers"))
)

# ── Final output ──────────────────────────────────────────────────────────────
df_final = (
    df_cross_agg
    .join(df_cinema_agg,
          (df_cross_agg.period         == df_cinema_agg.period) &
          (df_cross_agg.cin_title_name == df_cinema_agg.title_name),
          how="left")
    .drop(df_cinema_agg.period)
    .withColumn("LBE_Conversion_Rate",
        F.round(F.col("Cross_Visitors") / F.col("Title_Viewers") * 100, 1))
    .orderBy("cin_title_name", "LC_Trade_Brand_Description", "period", "days_bucket")
)

# df_final.display()

In [0]:
# ── Load proximity table ──────────────────────────────────────────────────────
df_prox = spark.table("data_exploration_hub.marketing_hub.lbe_location_distance") \
    .select(
        F.col("LBE_ID").cast("integer"),
        F.col("Theater_ID").cast("integer"),
        F.col("Distance_Band")
    )

# ── Enrich cross-nearest with distance band ───────────────────────────────────
df_cross_banded = (
    df_cross_nearest
    .join(df_prox,
          (df_cross_nearest.LocationId          == df_prox.LBE_ID) &
          (df_cross_nearest.cinema_location_id  == df_prox.Theater_ID),
          how="inner")
)

# ── Cross-visit metrics by band + day bucket ──────────────────────────────────
df_cross_agg = (
    df_cross_banded
    .groupBy("period", "cin_title_name", "LC_Trade_Brand_Description", "LBE_ID", "Distance_Band", "days_bucket")
    .agg(
        F.countDistinct("CDE_ID").alias("Cross_Visitors"),
        F.round(F.avg("days_gap"), 1).alias("Avg_Days_Cinema_to_LBE")
    )
)

# ── Title viewers scoped to catchment cinemas per LBE + band ─────────────────
df_catchment_viewers = (
    df_cinema_anchor
    .join(
        df_prox.select("LBE_ID", "Theater_ID", "Distance_Band"),
        df_cinema_anchor.cinema_location_id == df_prox.Theater_ID,
        how="inner"
    )
    .groupBy("period", "title_name", "LBE_ID", "Distance_Band")
    .agg(F.countDistinct("CDE_ID").alias("Title_Viewers"))
)

# ── Join catchment viewers onto cross agg ─────────────────────────────────────
df_final = (
    df_cross_agg
    .join(
        df_catchment_viewers,
        (df_cross_agg.period         == df_catchment_viewers.period) &
        (df_cross_agg.cin_title_name == df_catchment_viewers.title_name) &
        (df_cross_agg.LBE_ID         == df_catchment_viewers.LBE_ID) &
        (df_cross_agg.Distance_Band  == df_catchment_viewers.Distance_Band),
        how="left"
    )
    .drop(df_catchment_viewers.period)
    .drop(df_catchment_viewers.LBE_ID)
    .drop(df_catchment_viewers.Distance_Band)
    .drop("title_name")
    .withColumn("LBE_Conversion_Rate",
        F.round(F.col("Cross_Visitors") / F.col("Title_Viewers") * 100, 1))
    .orderBy("cin_title_name", "LC_Trade_Brand_Description", "Distance_Band", "period", "days_bucket")
)

# df_final.display()

In [0]:
# ── Collapse to brand level ───────────────────────────────────────────────────
df_cross_agg_brand = (
    df_cross_banded
    .groupBy("period", "cin_title_name", "LC_Trade_Brand_Description", "Distance_Band", "days_bucket")
    .agg(
        F.countDistinct("CDE_ID").alias("Cross_Visitors"),
        F.round(F.avg("days_gap"), 1).alias("Avg_Days_Cinema_to_LBE")
    )
)

# ── Total cross visitors per period + title + brand (across all bands) ────────
w_total = Window.partitionBy("period", "cin_title_name", "LC_Trade_Brand_Description")

df_final_share = (
    df_cross_agg_brand
    .withColumn("Total_Cross_Visitors", F.sum("Cross_Visitors").over(w_total))
    .withColumn("Band_Share_Pct",
        F.round(F.col("Cross_Visitors") / F.col("Total_Cross_Visitors") * 100, 1))
    .orderBy("cin_title_name", "LC_Trade_Brand_Description", "period", "Distance_Band", "days_bucket")
)

# df_final_share.display()

In [0]:
    df_film_att = (
    T_trans
    .join(T_locations.select("LocationID", "LC_Location_Type_Description", "LC_Trade_Brand_Description"), T_trans.LocationId == T_locations.LocationID, how='left')
    .join(T_date.select("dateid", "datefull"), T_trans.VisitDateId == T_date.dateid, how='left')
    .join(T_film.select("film_id", "title_id", "title_name"), T_film.film_id == T_trans.film_id, how='left')
    .filter(F.col("title_id") == '36120')
    .filter(F.col("TicketFLAG") == "Yes")
    .filter(F.col("SCENEFLAG") == 'Yes')
    .filter((F.col("VisitDateId") > '20250701') & (F.col("VisitDateId") < '20250930'))
    .groupBy("title_id")
    .agg(
        F.sum(F.col("Quantity")).alias("total_Tickets")
    )
    )

In [0]:
w_first = Window.partitionBy("CDE_ID", "cin_title_name", "LC_Trade_Brand_Description", "period")

df_cross_banded_first = (
    df_cross_banded
    .withColumn("rank", F.rank().over(w_first.orderBy("datefull")))
    .filter(F.col("rank") == 1)
    .drop("rank")
)
# ── Step 1: Isolate cross-visit keys from df_cross_banded ────────────────────
df_cross_keys = df_cross_banded.select(
    "CDE_ID",
    "datefull",
    F.col("LocationId").alias("lbe_location_id"),
    "period",
    "cin_title_name",
    F.col("LC_Trade_Brand_Description").alias("lbe_brand"),
    "Distance_Band",
    "days_bucket"
).distinct()

# ── Step 2: Join back to raw LBE transactions for concession revenue ──────────
df_cross_revenue = (
    df_all_trans
    .filter(F.col("title_name").isNull())
    .filter(F.col("ConcessionFLAG") == "Yes")
    .join(
        df_cross_keys,
        (df_all_trans.CDE_ID     == df_cross_keys.CDE_ID) &
        (df_all_trans.datefull   == df_cross_keys.datefull) &
        (df_all_trans.LocationId == df_cross_keys.lbe_location_id),
        how="inner"
    )
    .drop(df_cross_keys.CDE_ID)
    .drop(df_cross_keys.datefull)
    .drop("LC_Trade_Brand_Description")          # drop df_all_trans version
    .withColumnRenamed("lbe_brand", "LC_Trade_Brand_Description")
)

# ── Step 3: Aggregate revenue ─────────────────────────────────────────────────
df_revenue_agg = (
    df_cross_revenue
    .groupBy("period", "cin_title_name", "LC_Trade_Brand_Description", "Distance_Band", "days_bucket")
    .agg(
        F.countDistinct("CDE_ID").alias("Cross_Visitors"),
        F.countDistinct("visit_key").alias("Cross_Visits"),
        F.round(F.sum("Net_Revenue"), 2).alias("Total_Revenue"),
        F.round(F.sum("Net_Revenue") / F.countDistinct("visit_key"), 2).alias("Avg_Revenue_Per_Visit"),
        F.round(F.sum("Net_Revenue") / F.countDistinct("CDE_ID"), 2).alias("Avg_Revenue_Per_Member")
    )
    .orderBy("cin_title_name", "LC_Trade_Brand_Description", "Distance_Band", "period", "days_bucket")
)

df_revenue_agg.display()

In [0]:
# ── Step 0: Flag prior LBE visitors (6 months before each period) ─────────────
df_prior_lbe_2024 = (
    T_trans
    .join(T_locations.select("LocationID", "LC_Location_Type_Description"),
          T_trans.LocationId == T_locations.LocationID, how='left')
    .join(T_date.select("dateid", "datefull"),
          T_trans.VisitDateId == T_date.dateid, how='left')
    .filter(F.col("LC_Location_Type_Description") == "Restaurant/Store/Games")
    .filter(F.col("SCENEFLAG") == "Yes")
    .filter(F.col("CDE_ID").isNotNull())
    .filter(F.col("datefull").between(
        F.lit("2024-01-01").cast("date"),
        F.lit("2024-06-30").cast("date")
    ))
    .select("CDE_ID")
    .distinct()
    .withColumn("period", F.lit("July 2024 – Sep 2024"))
)

df_prior_lbe_2025 = (
    T_trans
    .join(T_locations.select("LocationID", "LC_Location_Type_Description"),
          T_trans.LocationId == T_locations.LocationID, how='left')
    .join(T_date.select("dateid", "datefull"),
          T_trans.VisitDateId == T_date.dateid, how='left')
    .filter(F.col("LC_Location_Type_Description") == "Restaurant/Store/Games")
    .filter(F.col("SCENEFLAG") == "Yes")
    .filter(F.col("CDE_ID").isNotNull())
    .filter(F.col("datefull").between(
        F.lit("2025-01-01").cast("date"),
        F.lit("2025-06-30").cast("date")
    ))
    .select("CDE_ID")
    .distinct()
    .withColumn("period", F.lit("July 2025 – Sep 2025"))
)

df_prior_lbe = df_prior_lbe_2024.union(df_prior_lbe_2025) \
    .withColumn("prior_lbe_visitor", F.lit(True))

# ── Keep only first LBE visit per member per title per period ─────────────────
# ── Keep only first LBE visit per member per title per period ─────────────────
w_first = Window.partitionBy("CDE_ID", "cin_title_name", "LC_Trade_Brand_Description", "period")

df_cross_banded_first = (
    df_cross_banded
    .withColumn("rank", F.rank().over(w_first.orderBy("datefull")))
    .filter(F.col("rank") == 1)
    .drop("rank")
    .alias("cb")
    .join(
        df_prior_lbe.alias("pl"),
        (F.col("cb.CDE_ID") == F.col("pl.CDE_ID")) &
        (F.col("cb.period") == F.col("pl.period")),
        how="left"
    )
    .withColumn("visitor_segment",
        F.when(F.col("pl.prior_lbe_visitor") == True, F.lit("Existing LBE Visitor"))
         .otherwise(F.lit("New / Lapsed LBE Visitor"))
    )
    .drop(F.col("pl.CDE_ID"))
    .drop(F.col("pl.period"))
    .drop("prior_lbe_visitor")
)


# ── Step 1: Isolate cross-visit keys ──────────────────────────────────────────
df_cross_keys = df_cross_banded_first.select(
    "CDE_ID",
    "datefull",
    F.col("LocationId").alias("lbe_location_id"),
    "period",
    "cin_title_name",
    F.col("LC_Trade_Brand_Description").alias("lbe_brand"),
    "Distance_Band",
    "days_bucket",
    "visitor_segment"
).distinct()

# ── Step 2: Join back to raw LBE transactions for concession revenue ──────────
df_cross_revenue = (
    df_all_trans
    .filter(F.col("title_name").isNull())
    .filter(F.col("ConcessionFLAG") == "Yes")
    .alias("at")
    .join(
        df_cross_keys.alias("ck"),
        (F.col("at.CDE_ID")     == F.col("ck.CDE_ID")) &
        (F.col("at.datefull")   == F.col("ck.datefull")) &
        (F.col("at.LocationId") == F.col("ck.lbe_location_id")),
        how="inner"
    )
    .drop(F.col("ck.CDE_ID"))
    .drop(F.col("ck.datefull"))
    .drop(F.col("at.LC_Trade_Brand_Description"))
    .withColumnRenamed("lbe_brand", "LC_Trade_Brand_Description")
)
# ── Step 3: Aggregate revenue by segment ──────────────────────────────────────
df_revenue_agg = (
    df_cross_revenue
    .groupBy("period", "cin_title_name", "LC_Trade_Brand_Description",
             "Distance_Band", "days_bucket", "visitor_segment")
    .agg(
        F.countDistinct("CDE_ID").alias("Cross_Visitors"),
        F.countDistinct("visit_key").alias("Cross_Visits"),
        F.round(F.sum("Net_Revenue"), 2).alias("Total_Revenue"),
        F.round(F.sum("Net_Revenue") / F.countDistinct("visit_key"), 2).alias("Avg_Revenue_Per_Visit"),
        F.round(F.sum("Net_Revenue") / F.countDistinct("CDE_ID"), 2).alias("Avg_Revenue_Per_Member")
    )
    .orderBy("cin_title_name", "LC_Trade_Brand_Description", "Distance_Band",
             "period", "visitor_segment", "days_bucket")
)

# ── Step 4: Cannibalization summary ───────────────────────────────────────────
w_total = Window.partitionBy("period", "cin_title_name", "LC_Trade_Brand_Description")

df_cannibalization = (
    df_revenue_agg
    .groupBy("period", "cin_title_name", "LC_Trade_Brand_Description", "visitor_segment")
    .agg(
        F.sum("Cross_Visitors").alias("Visitors"),
        F.round(F.sum("Total_Revenue"), 2).alias("Total_Revenue"),
        F.round(F.avg("Avg_Revenue_Per_Visit"), 2).alias("Avg_Revenue_Per_Visit")
    )
    .withColumn("Total_Cross_Visitors", F.sum("Visitors").over(w_total))
    .withColumn("Segment_Share_Pct",
        F.round(F.col("Visitors") / F.col("Total_Cross_Visitors") * 100, 1))
    .orderBy("cin_title_name", "LC_Trade_Brand_Description", "period", "visitor_segment")
)

df_revenue_agg.display()
df_cannibalization.display()